# Voice Clone Colab Remote GPU Backend

Run each cell from top to bottom. When the Cloudflare Tunnel URL appears, paste the `https://*.trycloudflare.com` URL into the desktop app's **Cấu hình** screen. Keep this notebook tab open while generating audio.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg curl
!pip install -q --no-warn-conflicts "fastapi==0.115.6" "starlette==0.41.3" "uvicorn[standard]==0.32.1" "pydantic>=2,<3" "huggingface_hub>=0.24,<1" python-multipart ebooklib pypdf beautifulsoup4
# Real F5-TTS stack. This must install successfully for real voice generation.
!pip install -q git+https://github.com/SWivid/F5-TTS.git

In [ ]:
import os, secrets
VOICE_CLONE_SHARED_SECRET = secrets.token_urlsafe(18)
os.environ['VOICE_CLONE_SHARED_SECRET'] = VOICE_CLONE_SHARED_SECRET
os.environ['VOICE_CLONE_BACKEND_MODE'] = 'colab'
os.environ['VOICE_CLONE_HOST'] = '127.0.0.1'
os.environ['VOICE_CLONE_PORT'] = '8787'
print('Paste this Shared secret into the desktop app:')
print(VOICE_CLONE_SHARED_SECRET)

In [ ]:
%%writefile server.py
from __future__ import annotations

import math
import mimetypes
import os
import random
import shutil
import subprocess
import sys
import threading
import time
import traceback
import uuid
import wave
from pathlib import Path
from typing import Any

from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

DATA = Path("/content/voice_clone_data")
UPLOADS = DATA / "uploads"
OUTPUTS = DATA / "outputs"
MODELS = DATA / "models"
for path in (UPLOADS, OUTPUTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

SHARED_SECRET = os.getenv("VOICE_CLONE_SHARED_SECRET", "")
DEFAULT_MODEL_ID = os.getenv("VOICE_CLONE_MODEL_ID", "hynt/F5-TTS-Vietnamese-ViVoice")
active_model: str | None = None
voices: list[dict[str, Any]] = [
    {
        "id": "colab-demo-vi",
        "name": "Colab Demo Vietnamese",
        "language": "vi",
        "kind": "system",
        "reference_path": None,
        "reference_url": None,
        "has_transcript": False,
        "usable": False,
    }
]
history: list[dict[str, Any]] = []
jobs: dict[str, dict[str, Any]] = {}
settings = {"asr_model": "base", "f5_precision": "auto"}

MODEL_REGISTRY = [
    {
        "id": "hynt/F5-TTS-Vietnamese-ViVoice",
        "name": "F5-TTS Vietnamese ViVoice",
        "license": "CC-BY-NC-SA-4.0",
        "languages": ["vi"],
        "vram_hint": "4-8GB",
        "adapter": "f5-tts",
        "local": True,
        "colab": True,
    },
    {
        "id": "SWivid/F5-TTS",
        "name": "F5-TTS Base",
        "license": "Model license varies by checkpoint",
        "languages": ["en", "multi"],
        "vram_hint": "6-8GB",
        "adapter": "f5-tts",
        "local": True,
        "colab": True,
    },
    {
        "id": "coqui/XTTS-v2",
        "name": "XTTS-v2",
        "license": "Coqui Public Model License",
        "languages": ["multi"],
        "vram_hint": "4-8GB",
        "adapter": "xtts",
        "local": True,
        "colab": True,
    },
]


class ModelDownloadRequest(BaseModel):
    model_id: str = DEFAULT_MODEL_ID


class GenerateRequest(BaseModel):
    text: str = ""
    voice_id: str = "colab-demo-vi"
    voice_name: str = "Colab Demo Vietnamese"
    language: str = "vi"
    speed: float = Field(1.0, ge=0.5, le=1.8)
    denoise: bool = True
    create_srt: bool = False
    output_format: str = "wav"
    nfe_step: int = 32
    cfg_strength: float = 2.0
    sway_sampling_coef: float = -1.0
    remove_silence: bool = False


class SrtGenerateRequest(GenerateRequest):
    segments: list[dict[str, Any]] = Field(default_factory=list)
    srt_text: str | None = None


class DialogueLineRequest(BaseModel):
    voice_id: str
    voice_name: str
    text: str


class DialogueGenerateRequest(GenerateRequest):
    lines: list[DialogueLineRequest] = Field(default_factory=list)


class SettingsRequest(BaseModel):
    asr_model: str | None = None
    f5_precision: str | None = None


app = FastAPI(title="Voice Clone Colab Backend", version="0.1.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])


@app.middleware("http")
async def require_secret(request: Request, call_next):
    if SHARED_SECRET and request.method != "OPTIONS" and request.url.path != "/health":
        provided = request.headers.get("x-voice-clone-secret") or request.query_params.get("vc_secret")
        if provided != SHARED_SECRET:
            raise HTTPException(status_code=401, detail="Shared secret khong hop le.")
    return await call_next(request)


def has_cuda() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False


def model_path(model_id: str) -> Path:
    return MODELS / ("models--" + model_id.replace("/", "--"))


def model_cached(model_id: str) -> bool:
    path = model_path(model_id)
    snapshots = path / "snapshots"
    if snapshots.exists() and any(snapshots.iterdir()):
        return True
    return path.exists() and any(item.is_file() for item in path.rglob("*") if item.name != "download-warning.txt")


def folder_size(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())


def format_ok(value: str) -> str:
    fmt = value.lower()
    if fmt not in {"wav", "mp3", "flac"}:
        raise HTTPException(status_code=400, detail="Dinh dang audio khong ho tro.")
    return fmt


def write_tone(path: Path, text: str, speed: float = 1.0) -> float:
    duration = max(1.0, min(90.0, len(text.split()) / max(1.0, 2.8 * speed)))
    sample_rate = 24000
    frames = int(duration * sample_rate)
    with wave.open(str(path), "w") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)
        for i in range(frames):
            env = min(1.0, i / max(1, sample_rate // 12), (frames - i) / max(1, sample_rate // 12))
            sample = int(9000 * env * math.sin(2 * math.pi * 180 * i / sample_rate))
            wav.writeframesraw(sample.to_bytes(2, "little", signed=True))
    return duration


def find_first(root: Path, patterns: list[str]) -> Path | None:
    if not root.exists():
        return None
    for pattern in patterns:
        matches = sorted(root.rglob(pattern))
        if matches:
            return matches[0]
    return None


def model_snapshot_dir(model_id: str) -> Path | None:
    snapshots = model_path(model_id) / "snapshots"
    if snapshots.exists():
        children = [child for child in snapshots.iterdir() if child.is_dir()]
        if children:
            return sorted(children, key=lambda item: item.stat().st_mtime, reverse=True)[0]
    return model_path(model_id) if model_path(model_id).exists() else None


def find_checkpoint(model_id: str) -> Path | None:
    root = model_snapshot_dir(model_id)
    if root is None:
        return None
    return find_first(root, ["*.safetensors", "*.pt", "*.pth", "*.bin"])


def find_vocab(model_id: str) -> Path | None:
    root = model_snapshot_dir(model_id)
    if root is None:
        return None
    return find_first(root, ["vocab.txt", "*.vocab", "*vocab*", "config.json"])


def find_f5_cli() -> str | None:
    for name in ("f5-tts_infer-cli", "f5_tts_infer_cli"):
        found = shutil.which(name)
        if found:
            return found
    for candidate in (Path(sys.executable).parent / "f5-tts_infer-cli", Path(sys.executable).parent / "f5_tts_infer_cli"):
        if candidate.exists():
            return str(candidate)
    return None


def normalize_reference_audio(source: Path, target: Path) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-ac", "1", "-ar", "24000", "-sample_fmt", "s16", str(target)],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr or "ffmpeg could not normalize reference audio")
    return target


def probe_audio_duration(path: Path) -> float | None:
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
            capture_output=True,
            text=True,
            timeout=20,
        )
        if result.returncode == 0 and result.stdout.strip():
            return float(result.stdout.strip())
    except Exception:
        return None
    return None


F5_RUNNER_SCRIPT = r"""
from __future__ import annotations

import json
import sys
from importlib.resources import files
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    CFM,
    get_tokenizer,
    infer_process,
    load_checkpoint,
    load_vocoder,
    n_fft,
    n_mel_channels,
    hop_length,
    ode_method,
    preprocess_ref_audio_text,
    remove_silence_for_generated_wav,
    target_sample_rate,
    win_length,
)
from f5_tts.model import DiT, UNetT


def checkpoint_dtype(precision: str, device: str):
    if precision == "fp16" and device == "cuda":
        return None
    return torch.float32


def load_f5_model_with_precision(model_cls, model_cfg, ckpt_file, vocab_file, device, precision):
    vocab_file = vocab_file or str(files("f5_tts").joinpath("infer/examples/vocab.txt"))
    vocab_char_map, vocab_size = get_tokenizer(vocab_file, "custom")
    model = CFM(
        transformer=model_cls(**model_cfg, text_num_embeds=vocab_size, mel_dim=n_mel_channels),
        mel_spec_kwargs=dict(
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mel_channels=n_mel_channels,
            target_sample_rate=target_sample_rate,
            mel_spec_type="vocos",
        ),
        odeint_kwargs=dict(method=ode_method),
        vocab_char_map=vocab_char_map,
    ).to(device)
    return load_checkpoint(model, ckpt_file, device, dtype=checkpoint_dtype(precision, device), use_ema=True)


def main() -> None:
    request = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = request.get("model_name") or "F5TTS_Base"
    model_cfg = OmegaConf.load(str(files("f5_tts").joinpath(f"configs/{model_name}.yaml")))
    model_node = getattr(model_cfg, "model", model_cfg)
    model_cls = {"DiT": DiT, "UNetT": UNetT}[model_node.backbone]
    vocoder = load_vocoder(vocoder_name="vocos", device=device)
    model = load_f5_model_with_precision(model_cls, model_node.arch, request["ckpt_file"], request.get("vocab_file") or "", device, request.get("precision") or "fp32")
    ref_audio, ref_text = preprocess_ref_audio_text(request["ref_audio"], request["ref_text"])
    audio, sample_rate, _ = infer_process(
        ref_audio,
        ref_text,
        request["gen_text"],
        model,
        vocoder,
        mel_spec_type="vocos",
        nfe_step=int(request.get("nfe_step") or 32),
        cfg_strength=float(request.get("cfg_strength") or 2.0),
        sway_sampling_coef=float(request.get("sway_sampling_coef") or -1.0),
        speed=float(request.get("speed") or 1.0),
        device=device,
    )
    output = Path(request["output"])
    output.parent.mkdir(parents=True, exist_ok=True)
    audio = np.asarray(audio, dtype=np.float32)
    nan_count = int(np.isnan(audio).sum()) if audio.size else 0
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
    max_abs = float(np.max(np.abs(audio))) if audio.size else 0.0
    mean_abs = float(np.mean(np.abs(audio))) if audio.size else 0.0
    print(f"audio_stats size={audio.size} nan={nan_count} max_abs={max_abs} mean_abs={mean_abs}", flush=True)
    if max_abs <= 1e-4 or mean_abs <= 1e-6:
        raise RuntimeError("F5-TTS produced silent audio")
    if max_abs > 1.0:
        audio = audio / max_abs * 0.95
    sf.write(str(output), audio, sample_rate)
    if request.get("remove_silence"):
        remove_silence_for_generated_wav(str(output))
    print(str(output), flush=True)


if __name__ == "__main__":
    main()
"""


def try_f5_tts(job_id: str, text: str, payload: GenerateRequest, output_path: Path) -> tuple[Path, float]:
    if not active_model:
        raise HTTPException(status_code=400, detail="Colab chua tai model. Hay bam Tai model truoc khi tao giong.")
    voice = next((item for item in voices if item["id"] == payload.voice_id), None)
    if not voice or not voice.get("reference_path"):
        raise HTTPException(status_code=400, detail="Giong nay chua co audio mau tren Colab.")
    reference_text = str(voice.get("transcript") or "").strip()
    if not reference_text:
        raise HTTPException(status_code=400, detail="Giong nay chua co transcript mau. Hay upload voice kem transcript.")
    reference_path = Path(str(voice["reference_path"]))
    if not reference_path.exists():
        raise HTTPException(status_code=400, detail="Khong tim thay file audio mau tren Colab.")
    output_dir = OUTPUTS / f"{job_id}-f5"
    output_dir.mkdir(parents=True, exist_ok=True)
    normalized_ref = normalize_reference_audio(reference_path, output_dir / "reference_24k.wav")
    ckpt = find_checkpoint(active_model)
    vocab = find_vocab(active_model)
    if not ckpt:
        raise HTTPException(status_code=500, detail="Khong tim thay checkpoint F5-TTS trong model Colab.")
    runner_path = output_dir / "f5_runner.py"
    request_path = output_dir / "f5_request.json"
    generated_path = output_dir / "generated.wav"
    runner_path.write_text(F5_RUNNER_SCRIPT, encoding="utf-8")
    request_path.write_text(
        json.dumps(
            {
                "ref_audio": str(normalized_ref),
                "ref_text": reference_text,
                "gen_text": text,
                "output": str(generated_path),
                "ckpt_file": str(ckpt),
                "vocab_file": str(vocab) if vocab else "",
                "model_name": "F5TTS_Base",
                "speed": payload.speed,
                "nfe_step": payload.nfe_step,
                "cfg_strength": payload.cfg_strength,
                "sway_sampling_coef": payload.sway_sampling_coef,
                "remove_silence": payload.remove_silence,
                "precision": "fp32" if settings.get("f5_precision", "auto") == "auto" else settings.get("f5_precision", "fp32"),
            },
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    command = [sys.executable, str(runner_path), str(request_path)]
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUTF8"] = "1"
    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1800, env=env)
    (output_dir / "f5_cli.log").write_text(
        "COMMAND:\n" + " ".join(command) + "\n\nRETURN_CODE:\n" + str(result.returncode) + "\n\nSTDOUT:\n" + (result.stdout or "") + "\n\nSTDERR:\n" + (result.stderr or ""),
        encoding="utf-8",
        errors="ignore",
    )
    if result.returncode != 0 or not generated_path.exists():
        raise HTTPException(status_code=500, detail=f"F5-TTS khong tao duoc audio that. Xem log trong Colab: {output_dir / 'f5_cli.log'}")
    shutil.copyfile(generated_path, output_path)
    duration = probe_audio_duration(output_path) or max(1.0, len(text.split()) / 2.6)
    return output_path, duration

def convert_audio(source: Path, job_id: str, fmt: str) -> Path:
    if fmt == "wav":
        return source
    target = OUTPUTS / f"{job_id}.{fmt}"
    result = subprocess.run(["ffmpeg", "-y", "-i", str(source), str(target)], capture_output=True, text=True)
    if result.returncode != 0:
        return source
    return target


def output_response(path: Path, download: bool = False) -> FileResponse:
    media_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    if download:
        return FileResponse(path, filename=path.name, media_type=media_type)
    return FileResponse(path, media_type=media_type, headers={"Content-Disposition": f'inline; filename="{path.name}"'})


def save_history(job_id: str, voice_name: str, text: str, duration: float, path: Path, seed: int, mock: bool) -> dict[str, Any]:
    item = {
        "id": job_id,
        "voice_name": voice_name,
        "text_preview": text[:180],
        "duration_seconds": duration,
        "format": path.suffix.lstrip(".").upper(),
        "language": "Tieng Viet",
        "seed": seed,
        "output_url": f"/outputs/{path.name}",
        "created_at": "Colab session",
        "mock": mock,
    }
    history.insert(0, item)
    return {
        "job_id": job_id,
        "output_url": item["output_url"],
        "duration_seconds": duration,
        "format": item["format"],
        "seed": seed,
        "mock": mock,
        "message": "Colab da tao audio.",
    }


@app.get("/health")
def health():
    cuda = has_cuda()
    return {"ok": True, "api_version": "0.1.0", "backend": "colab-adapter", "device": "CUDA" if cuda else "CPU", "cuda": cuda, "model_loaded": active_model, "mode": "colab", "warning": None if active_model else "Colab chua tai model trong session nay."}


@app.get("/capabilities")
def capabilities():
    return {"audio_formats": ["wav", "mp3", "flac"], "inputs": ["text", "srt"], "backends": ["colab"], "models": MODEL_REGISTRY}


@app.get("/models")
def get_models():
    return {"items": [{**m, "cached": model_cached(m["id"]), "path": str(model_path(m["id"])), "size_bytes": folder_size(model_path(m["id"])), "checkpoint": None, "vocab": None} for m in MODEL_REGISTRY], "active": active_model, "cache_root": str(MODELS), "default_model_id": DEFAULT_MODEL_ID}


@app.post("/models/download")
def download_model(payload: ModelDownloadRequest):
    global active_model
    ids = {m["id"] for m in MODEL_REGISTRY}
    if payload.model_id not in ids:
        raise HTTPException(status_code=404, detail="Model khong nam trong registry.")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=payload.model_id, cache_dir=str(MODELS), local_dir_use_symlinks=False)
    except Exception as exc:
        target = model_path(payload.model_id)
        target.mkdir(parents=True, exist_ok=True)
        (target / "download-warning.txt").write_text(str(exc), encoding="utf-8", errors="ignore")
        raise HTTPException(status_code=500, detail=f"Khong tai duoc model Hugging Face: {exc}") from exc
    if not model_cached(payload.model_id):
        raise HTTPException(status_code=500, detail="Da goi snapshot_download nhung khong tim thay cache model trong Colab.")
    active_model = payload.model_id
    return {"ok": True, "model_id": payload.model_id, "cached": True, "path": str(model_path(payload.model_id)), "size_bytes": folder_size(model_path(payload.model_id))}


@app.delete("/models/{model_id:path}")
def delete_model(model_id: str):
    global active_model
    path = model_path(model_id)
    if path.exists():
        shutil.rmtree(path)
    if active_model == model_id:
        active_model = None
    return {"ok": True, "model_id": model_id}


@app.get("/settings")
def get_settings():
    return {
        "asr_model": settings["asr_model"],
        "asr_models": ["tiny", "base", "small", "medium", "large-v3-turbo"],
        "f5_precision": settings.get("f5_precision", "auto"),
        "f5_precision_options": ["auto", "fp32", "fp16"],
    }


@app.post("/settings")
def update_settings(payload: SettingsRequest):
    if payload.asr_model:
        settings["asr_model"] = payload.asr_model
    if payload.f5_precision:
        precision = payload.f5_precision.lower()
        if precision not in {"auto", "fp32", "fp16"}:
            raise HTTPException(status_code=400, detail="F5 precision khong hop le.")
        settings["f5_precision"] = precision
    return get_settings()


@app.get("/voices")
def get_voices():
    return {"items": voices}


@app.post("/voices/upload")
async def upload_voice(name: str = Form(...), language: str = Form("vi"), transcript: str = Form(""), consent: bool = Form(False), file: UploadFile = File(...)):
    if not consent:
        raise HTTPException(status_code=400, detail="Can xac nhan quyen su dung giong.")
    voice_id = uuid.uuid4().hex[:12]
    suffix = Path(file.filename or "voice.wav").suffix or ".wav"
    path = UPLOADS / f"{voice_id}{suffix}"
    path.write_bytes(await file.read())
    item = {"id": voice_id, "name": name, "language": language, "kind": "user", "reference_path": str(path), "reference_url": f"/voices/{voice_id}/audio", "has_transcript": bool(transcript.strip()), "usable": True, "transcript": transcript}
    voices.append(item)
    return item


@app.get("/voices/{voice_id}/audio")
def voice_audio(voice_id: str, download: bool = False):
    voice = next((v for v in voices if v["id"] == voice_id), None)
    if not voice or not voice.get("reference_path"):
        raise HTTPException(status_code=404, detail="Khong co audio mau.")
    return output_response(Path(voice["reference_path"]), download)


@app.delete("/voices/{voice_id}")
def delete_voice(voice_id: str):
    global voices
    voices = [v for v in voices if v["id"] != voice_id or v.get("kind") == "system"]
    return {"ok": True}


@app.post("/generate/text")
def generate_text(payload: GenerateRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Van ban trong.")
    fmt = format_ok(payload.output_format)
    job_id = uuid.uuid4().hex[:16]
    seed = random.randint(100000000, 999999999)
    wav_path = OUTPUTS / f"{job_id}.wav"
    if active_model:
        real_path, duration = try_f5_tts(job_id, text, payload, wav_path)
        final = convert_audio(real_path, job_id, fmt)
        return save_history(job_id, payload.voice_name, text, duration, final, seed, False)
    raise HTTPException(status_code=400, detail="Colab chua tai model. Hay bam Tai model truoc khi tao giong.")


@app.post("/generate/srt")
def generate_srt(payload: SrtGenerateRequest):
    text = "\n".join(str(s.get("text", "")) for s in payload.segments).strip() or payload.text
    return generate_text(GenerateRequest(**{**payload.dict(), "text": text}))


@app.post("/generate/dialogue")
def generate_dialogue(payload: DialogueGenerateRequest):
    text = "\n".join(f"{line.voice_name}: {line.text}" for line in payload.lines if line.text.strip())
    return generate_text(GenerateRequest(**{**payload.dict(), "text": text, "voice_name": "Hoi thoai"}))


def run_generate_job(job_id: str, kind: str, payload_data: dict[str, Any]) -> None:
    jobs[job_id].update({"status": "running", "updated_at": time.time()})
    try:
        if kind == "text":
            result = generate_text(GenerateRequest(**payload_data))
        elif kind == "srt":
            result = generate_srt(SrtGenerateRequest(**payload_data))
        elif kind == "dialogue":
            result = generate_dialogue(DialogueGenerateRequest(**payload_data))
        else:
            raise HTTPException(status_code=400, detail="Loai job khong hop le.")
        jobs[job_id].update({"status": "done", "result": result, "updated_at": time.time()})
    except HTTPException as exc:
        jobs[job_id].update({"status": "error", "error": str(exc.detail), "updated_at": time.time()})
    except Exception as exc:
        jobs[job_id].update({"status": "error", "error": str(exc), "traceback": traceback.format_exc()[-4000:], "updated_at": time.time()})


@app.post("/jobs/generate/{kind}")
def start_generate_job(kind: str, payload: dict[str, Any]):
    if kind not in {"text", "srt", "dialogue"}:
        raise HTTPException(status_code=400, detail="Loai job khong hop le.")
    job_id = uuid.uuid4().hex[:16]
    jobs[job_id] = {"job_id": job_id, "status": "queued", "kind": kind, "created_at": time.time(), "updated_at": time.time()}
    thread = threading.Thread(target=run_generate_job, args=(job_id, kind, payload), daemon=True)
    thread.start()
    return jobs[job_id]


@app.get("/jobs/{job_id}")
def get_generate_job(job_id: str):
    item = jobs.get(job_id)
    if not item:
        raise HTTPException(status_code=404, detail="Khong tim thay job.")
    return item


@app.get("/history")
def get_history(page: int = 1, page_size: int = 6):
    page = max(1, page)
    page_size = max(1, min(50, page_size))
    start = (page - 1) * page_size
    total = len(history)
    pages = max(1, math.ceil(total / page_size))
    return {"items": history[start:start + page_size], "page": page, "pages": pages, "total": total}


@app.delete("/history/{job_id}")
def delete_history(job_id: str):
    global history
    history = [item for item in history if item["id"] != job_id]
    for path in OUTPUTS.glob(f"{job_id}.*"):
        path.unlink(missing_ok=True)
    return {"ok": True}


@app.get("/outputs/{filename}")
def outputs(filename: str, download: bool = False):
    path = (OUTPUTS / filename).resolve()
    if OUTPUTS not in path.parents and path != OUTPUTS:
        raise HTTPException(status_code=400, detail="Duong dan output khong hop le.")
    if not path.exists():
        raise HTTPException(status_code=404, detail="Khong tim thay output.")
    return output_response(path, download)


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8787)


In [ ]:
import os, subprocess, time, urllib.request
subprocess.run('fuser -k 8787/tcp || true', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
server = subprocess.Popen(['python', 'server.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for attempt in range(30):
    if server.poll() is not None:
        output = server.stdout.read() if server.stdout else ''
        raise RuntimeError('Backend exited early:\n' + output[-4000:])
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8787/health', timeout=2).read().decode())
        print('Backend started on http://127.0.0.1:8787')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Backend did not become healthy on port 8787')

In [ ]:
!curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, re, time, urllib.request
print(urllib.request.urlopen('http://127.0.0.1:8787/health', timeout=5).read().decode())
subprocess.run('pkill -f cloudflared || true', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Waiting for Cloudflare Tunnel URL...')
for _ in range(80):
    line = proc.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        print('\nPaste this Colab URL into the desktop app:')
        print(match.group(0))
        print('\nShared secret:')
        print(VOICE_CLONE_SHARED_SECRET)
        break
